In [1]:
import pandas as pd
import os

# 파일 경로
input_file = os.path.join(os.getcwd(), 'authoress_original.xlsx')
output_file = os.path.join(os.getcwd(), 'authoress.xlsx')

# Contents 전용 삭제 키워드 (Title, Contents 둘 다 적용)  '대입', '입시', '수능', '학종', '학생부', '학생부종합', '내신', '특기자', '논술', '후보자','징계'
keywords_contents = [
     '대입', '입시', '수능', '학종', '학생부', '학생부종합', '내신', '특기자', '논술', '후보자','징계','후보',
]

# Title 전용 추가 삭제 키워드
keywords_title_only = [ '인사','대표','회장','임명','취임','부고','부음','분양','시티','오피스텔']

# 파일 읽기
df = pd.read_excel(input_file)

# 패턴 생성
pattern_contents = '|'.join([f'{kw.strip()}' for kw in keywords_contents])
pattern_title = '|'.join([f'{kw.strip()}' for kw in keywords_contents + keywords_title_only])

# Title 열 검사: 모든 키워드 (Title 전용 포함)
title_mask = df['Title'].astype(str).str.contains(pattern_title, na=False)

# Contents 열 검사: Contents 전용 키워드만
contents_mask = df['Contents'].astype(str).str.contains(pattern_contents, na=False)

# Title 또는 Contents 중 하나라도 키워드 포함된 행 (OR 조건)
combined_mask = title_mask | contents_mask

# 삭제 전 행 수
initial_count = len(df)

# 해당 행 삭제
df_filtered = df[~combined_mask]

# 삭제 후 행 수
final_count = len(df_filtered)

# 삭제된 행 수
deleted_count = initial_count - final_count

# 새로운 파일명으로 저장
df_filtered.to_excel(output_file, index=False)

# 결과 출력
print(f"Title 또는 Contents 열에서 {deleted_count}행이 삭제되었고, {final_count}행이 저장되었습니다.")
print(f"Title에서 {title_mask.sum()}행, Contents에서 {contents_mask.sum()}행이 매칭되었습니다. (중복 포함)")
print(f"Title 전용 키워드 '{keywords_title_only}'에서 {df['Title'].astype(str).str.contains('|'.join(keywords_title_only), na=False).sum()}행 매칭")


Title 또는 Contents 열에서 759행이 삭제되었고, 9089행이 저장되었습니다.
Title에서 229행, Contents에서 575행이 매칭되었습니다. (중복 포함)
Title 전용 키워드 '['인사', '대표', '회장', '임명', '취임', '부고', '부음', '분양', '시티', '오피스텔']'에서 193행 매칭


## 전처리 작업

In [2]:
import pandas as pd
import os
from collections import Counter


# ===== 1. 파일 경로 지정 =====
base_dir = os.getcwd()
input_file_path = os.path.join(base_dir, 'authoress.xlsx')
output_file_path = os.path.join(base_dir, 'authoress_Token.xlsx')



# ===== 2. 문자열 치환 사전 정의 =====
replace_dict = {

    '대한민국': '한국',
    '우리나라': '한국',
    '여자': '여성',
    '남자': '남성',
    '어린이': '아동',
    '사람': '인간',
    '사람들': '인간',
    '인간들': '인간',
    '여성들': '여성',
    '남성들': '남성',
    '아이들': '아동',
    '아이': '아동',
    '어린이': '아동',
    '아기들': '아기',
    '학생들': '학생',
    '강좌': '강연',
    '강의': '강연',
    '얘기': '이야기',
    '그림': '회화',
    '전시회':'전시',
    '음락계':'음악계',
    '루이나이웨이':'루이'

    


}


# ===== 3. 불용어 집합 정의 =====
stopwords = set([ 
    '경우', '특별', '생각', '이날', '최고', '과장', '분야', '결과', '주장', '적용', '2만', '10년',
    '예정', '입장', '회장', '요구', '발생', '대표', '센터', '전문', '목적', '상황', '의원', '시작', '상태',
    '위원', '참여', '사실', '여부', '확대', '팀장', '포함', '정도', '위원장', '판단', '구성', '중요', '본부',
    '수준', '생산', '자체', '원장', '위원회', '사이', '시대', '진행', '주제', '이해', '설명', '1만',
    '강조', '운영', '대상', '활동', '가능', '질문', '소개', '준비', '선택', '바탕', '마련', '가지',
    '현장', '선정', '영역', '결국', '등장', '주목', '핵심', '세기', '시절', '사용', '진행', '중심','제공',
    '역할', '총장', '접근', '만큼', '요즘', '유명', '자료', '사진', '그동안', '자리', '방식', '출신', 
    '방향', '마지막', '그동안', '차원', '관계자', '1년', '000원', '결정', '논의', '행위','사례','시행',
    '제기','정보','근거','실제','단계','기간','지난달','계기','편집','의미','현실','확인','방법','7시',
    '이름','영향','계획','사태','동시','기본','순간','마리','역할','활용','기준','추진','목소리','공개',
    '공동','작성','전망','모습','예상','과거','내년','혐의','능력','이용','존재','모습','의견','말씀','표현',
    '논란','사전','효과','특정','요청','소장','내달','vs', 'VS','9단','시론','각종','개최','항목','A씨','B씨','C씨',
    '선생',


    '여류'
])


# ===== 4. 변환 카운터 생성 =====
replace_counter = Counter()


# ===== 5. 토큰 단위 처리 함수 (유의어 치환 + 불용어 제거) =====
def process_tokens(text):
    """토큰화된 텍스트에서 유의어 치환 및 불용어 제거"""
    if pd.isnull(text):
        return ''
    
    if not isinstance(text, str):
        text = str(text)
    
    # 띄어쓰기로 분리된 토큰들을 리스트로 변환
    tokens = text.split()
    
    # 유의어 치환 (토큰 단위로 정확히 매칭)
    new_tokens = []
    for token in tokens:
        if token in replace_dict:
            new_token = replace_dict[token]
            replace_counter[token] += 1
            new_tokens.append(new_token)
        else:
            new_tokens.append(token)
    
    tokens = new_tokens
    
    # 불용어 제거
    tokens = [token for token in tokens if token not in stopwords]
    
    # 다시 띄어쓰기로 결합
    return ' '.join(tokens)


# ===== 6. 엑셀 파일 읽기 =====
print("파일 읽기 중...")
df = pd.read_excel(input_file_path)


# ===== 7. Token 열 생성 (유의어 치환 + 불용어 제거) =====
print("토큰 처리 중...")
total_rows = len(df)
save_interval = 5000

for i in range(0, total_rows, save_interval):
    end_index = min(i + save_interval, total_rows)
    df.loc[i:end_index-1, 'Token'] = df.loc[i:end_index-1, 'Contents'] \
                                      .fillna('') \
                                      .astype(str) \
                                      .apply(process_tokens)
    print(f"{end_index}건 처리 완료")


# ===== 8. 변환된 단어별 통계 출력 =====
print("\n--- 단어별 치환 횟수 ---")
if replace_counter:
    for word, count in replace_counter.most_common():
        print(f'"{word}" → "{replace_dict[word]}" : {count}회')
else:
    print("치환된 단어가 없습니다.")


# ===== 9. 최종 결과를 새 파일로 저장 =====
print("\n최종 결과 저장 중...")
df.to_excel(output_file_path, index=False)
print(f"✓ 총 {total_rows}행 처리됨")

파일 읽기 중...
토큰 처리 중...
5000건 처리 완료
9089건 처리 완료

--- 단어별 치환 횟수 ---
"사람" → "인간" : 4686회
"여자" → "여성" : 3616회
"그림" → "회화" : 2296회
"사람들" → "인간" : 2041회
"남자" → "남성" : 1715회
"전시회" → "전시" : 1082회
"얘기" → "이야기" : 909회
"루이나이웨이" → "루이" : 865회
"우리나라" → "한국" : 844회
"아이" → "아동" : 803회
"여성들" → "여성" : 729회
"아이들" → "아동" : 515회
"어린이" → "아동" : 478회
"대한민국" → "한국" : 337회
"학생들" → "학생" : 277회
"강의" → "강연" : 255회
"남성들" → "남성" : 188회
"강좌" → "강연" : 127회
"인간들" → "인간" : 60회
"아기들" → "아기" : 9회
"음락계" → "음악계" : 1회

최종 결과 저장 중...
✓ 총 9089행 처리됨


### 메타데이터 설정

In [3]:
import pandas as pd
import os


# 파일 경로
file_path = os.path.join(os.getcwd(), 'authoress_Token.xlsx')


# 단어 여러 개 한 줄에 입력 (쉼표로 구분)
keywords_input = "세계, 한국"  # 여기에 단어를 쉼표로 구분하여 입력하세요
keywords = [kw.strip() for kw in keywords_input.split(',') if kw.strip()]


# 입력이 없으면 종료
if not keywords:
    print("입력된 단어가 없어 작업을 종료합니다.")
    exit()


# 파일 읽기
df = pd.read_excel(file_path)


# 각 키워드별로 열 생성 및 값 할당 (Title 또는 Contents에 포함되면 1, 아니면 0)
for kw in keywords:
    df[kw] = (
        df['Token'].astype(str).str.contains(kw, na=False) |
        df['Title'].astype(str).str.contains(kw, na=False)
    ).astype(int)


# Voca열 생성: 값이 1인 키워드명을 '+'로 연결, 모두 0이면 '없음'
def make_voca(row):
    included = [kw for kw in keywords if row[kw] == 1]
    return '+'.join(included) if included else '없음'


df['Voca'] = df.apply(make_voca, axis=1)


# 같은 파일명으로 저장 (덮어쓰기)
df.to_excel(file_path, index=False)


print(f'작업이 완료되었습니다. 처리된 키워드: {", ".join(keywords)}')


작업이 완료되었습니다. 처리된 키워드: 세계, 한국


### "여류"가 없는 사례 삭제

In [4]:
import pandas as pd

# 파일명
file_name = 'authoress.xlsx'

# 엑셀 파일 읽기
df = pd.read_excel(file_name)

# 전체 행 수
total_rows = len(df)

# "여류"가 Title 또는 Contents 열에 포함된 행만 필터링 (대소문자 구분없이)
mask = df['Title'].str.contains("여류", case=False, na=False) | df['Contents'].str.contains("여류", case=False, na=False)
filtered_df = df[mask]

# 삭제된 행 수
deleted_rows = total_rows - len(filtered_df)

# 결과 출력
print(f"총 {total_rows}행 중 {deleted_rows}행이 삭제되었고, {len(filtered_df)}행이 남았습니다.")

# 같은 파일명으로 저장 (덮어쓰기)
filtered_df.to_excel(file_name, index=False)

총 9089행 중 23행이 삭제되었고, 9066행이 남았습니다.


## 빈도 및 TF-IDF

In [5]:
import pandas as pd
from collections import Counter
import os
from gensim.corpora import Dictionary
from gensim.models import TfidfModel

# ===== 추적할 단어 목록 (여기서 수정 가능) =====
# 빈 리스트일 경우 단어 추이 시트가 생성되지 않습니다
TRACKING_WORDS = ['세계','한국']
# TRACKING_WORDS = []  # 단어 추이를 생성하지 않으려면 이렇게 설정

file_path = 'authoress_Token.xlsx'


# ===== Excel 파일 읽기 (여러 방법 시도) =====
df = None
engines_to_try = ['openpyxl', 'xlrd', 'calamine']

for engine in engines_to_try:
    try:
        df = pd.read_excel(file_path, engine=engine)
        print(f"✓ {engine} 엔진으로 성공!")
        break
    except Exception as e:
        print(f"✗ {engine} 실패: {type(e).__name__}")
        continue


if df is None:
    print("\n❌ 모든 엔진 시도 실패")
    print("\n해결 방법:")
    print("1. Excel 파일을 다시 저장해보세요")
    print("2. 파일이 실제로 .xlsx 형식인지 확인하세요")
    print("3. 필요한 라이브러리를 설치하세요: pip install openpyxl xlrd calamine")
    exit()


print(f"\n✓ 데이터 로드 완료: {len(df)} 행")


output_path = os.path.join(os.path.dirname(file_path), 'authoress_Freq.xlsx')



with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    workbook = writer.book
    
    # 1. 연도별 발행처별 기사 수 (교차표)
    if 'Year' in df.columns and 'Publisher' in df.columns:
        # 연도와 발행처별 교차표 생성
        year_publisher_counts = pd.crosstab(df['Year'], df['Publisher'])
        year_publisher_counts = year_publisher_counts.sort_index()
        year_publisher_counts['합계'] = year_publisher_counts.sum(axis=1)
        
        # 행 추가: 전체 합계
        total_row = year_publisher_counts.sum(axis=0)
        total_row.name = '합계'
        year_publisher_counts = pd.concat([year_publisher_counts, pd.DataFrame([total_row])])
        
        year_publisher_counts.to_excel(writer, sheet_name='연도별 기사 수')
        worksheet_year = writer.sheets['연도별 기사 수']
        
        # 발행처별 라인 그래프 생성
        num_publishers = len(year_publisher_counts.columns) - 1  # '합계' 제외
        publishers = list(year_publisher_counts.columns[:-1])
        
        chart_positions = [f'H{2 + 18 * i}' for i in range(num_publishers)]
        
        for idx, publisher in enumerate(publishers):
            chart = workbook.add_chart({'type': 'line'})
            # 데이터 행: 첫 데이터 행(1)부터 마지막 데이터 행까지 (합계 행 제외)
            data_rows = len(year_publisher_counts) - 1
            chart.add_series({
                'name':       publisher,
                'categories': ['연도별 기사 수', 1, 0, data_rows, 0],
                'values':     ['연도별 기사 수', 1, idx+1, data_rows, idx+1],
                'marker':     {'type': 'circle', 'size': 7},
            })
            chart.set_title({'name': f'{publisher} 연도별 기사 수'})
            chart.set_x_axis({'name': 'Year'})
            chart.set_y_axis({'name': '기사 수'})
            worksheet_year.insert_chart(chart_positions[idx], chart)

        # 전체 합계 그래프 추가
        chart_total = workbook.add_chart({'type': 'line'})
        data_rows = len(year_publisher_counts) - 1
        chart_total.add_series({
            'name': '전체 합계',
            'categories': ['연도별 기사 수', 1, 0, data_rows, 0],
            'values': ['연도별 기사 수', 1, num_publishers + 1, data_rows, num_publishers + 1],
            'marker': {'type': 'circle', 'size': 7},
        })
        chart_total.set_title({'name': '전체 기사 수 추이 (모든 발행처 합계)'})
        chart_total.set_x_axis({'name': 'Year'})
        chart_total.set_y_axis({'name': '기사 수'})
        worksheet_year.insert_chart(f'H{2 + 18 * num_publishers}', chart_total)



    # 2. 발행처 별 기사 수 (별도 시트)
    if 'Publisher' in df.columns:
        publisher_counts_sep = df['Publisher'].value_counts().reset_index()
        publisher_counts_sep.columns = ['Publisher', '기사 수']
        publisher_counts_sep = publisher_counts_sep.sort_values(by='기사 수', ascending=False)
        publisher_counts_sep.to_excel(writer, sheet_name='발행처 별 기사 수', index=False)
        worksheet_pub = writer.sheets['발행처 별 기사 수']
        chart_pub = workbook.add_chart({'type': 'pie'})
        chart_pub.add_series({
            'name': '발행처 별 기사 수',
            'categories': ['발행처 별 기사 수', 1, 0, len(publisher_counts_sep), 0],
            'values': ['발행처 별 기사 수', 1, 1, len(publisher_counts_sep), 1],
            'data_labels': {'percentage': True},
        })
        chart_pub.set_title({'name': '발행처 별 기사 수'})
        worksheet_pub.insert_chart('D2', chart_pub)



    # 2-1. 용어 별 기사 추이 시트 (Token 열 뒤의 Voca 제외 모든 열 자동 인식)
    if 'Token' in df.columns:
        token_idx = df.columns.get_loc('Token')
        term_cols = [col for col in list(df.columns[token_idx+1:]) if col != 'Voca']



        if term_cols:
            term_yearly = df.groupby('Year')[term_cols].sum().astype(int).reset_index()
            total_row = ['합계'] + [term_yearly[col].sum() for col in term_cols]
            term_yearly_with_total = pd.concat([term_yearly, pd.DataFrame([total_row], columns=term_yearly.columns)], ignore_index=True)
            term_yearly_with_total.to_excel(writer, sheet_name='용어 별 기사 추이', index=False)
            worksheet_term = writer.sheets['용어 별 기사 추이']



            chart_positions = [f'F{2 + 16 * i}' for i in range(len(term_cols))]
            for idx, col in enumerate(term_cols):
                chart = workbook.add_chart({'type': 'line'})
                chart.add_series({
                    'name':       col,
                    'categories': ['용어 별 기사 추이', 1, 0, len(term_yearly), 0],
                    'values':     ['용어 별 기사 추이', 1, idx+1, len(term_yearly), idx+1],
                    'marker':     {'type': 'circle', 'size': 7},
                })
                chart.set_title({'name': f'{col} 연도별 기사 수'})
                chart.set_x_axis({'name': 'Year'})
                chart.set_y_axis({'name': '기사 수'})
                worksheet_term.insert_chart(chart_positions[idx], chart)



    # 3. 상위 300개 단어
    if 'Token' in df.columns:
        df['Token'] = df['Token'].fillna('').astype(str)
        tokens = df['Token'].str.cat(sep=' ').split()
        token_counts = Counter(tokens).most_common(300)



        token_lists = df['Token'].apply(lambda x: x.split())



        def calculate_doc_count(token):
            return token_lists.apply(lambda toks: token in toks).sum()



        token_counts_with_docs = [(tk, freq, calculate_doc_count(tk)) for tk, freq in token_counts]
        texts = [doc.split() for doc in df['Token'].dropna()]
        dictionary = Dictionary(texts)
        corpus = [dictionary.doc2bow(text) for text in texts]
        tfidf_model = TfidfModel(corpus)
        tfidf_scores = {dictionary[id]: score for doc in tfidf_model[corpus] for id, score in doc}
        token_df = pd.DataFrame(token_counts_with_docs, columns=['Token', 'Frequency', 'Doc Count'])
        token_df['TF-IDF'] = token_df['Token'].apply(lambda x: tfidf_scores.get(x, 0))
        token_df['Average Occurrences'] = token_df.apply(lambda row: row['Frequency'] / row['Doc Count'], axis=1)
        total_docs = df.shape[0]
        token_df['Doc Percentage'] = token_df['Doc Count'].apply(lambda x: round((x / total_docs) * 100, 2))
        token_df = token_df.head(300).sort_values(by='Frequency', ascending=False)
        token_df = token_df[['Token', 'Frequency', 'TF-IDF', 'Doc Count', 'Doc Percentage', 'Average Occurrences']]
        token_df.to_excel(writer, sheet_name='상위 300', index=False)



        worksheet_token = writer.sheets['상위 300']
        (max_row, max_col) = token_df.shape
        worksheet_token.autofilter(0, 0, max_row, max_col - 1)



    # 4. 연도별 상위 20개 단어
    if 'Year' in df.columns and 'Token' in df.columns:
        yearly_top_20_tokens = df.groupby('Year')['Token'].apply(
            lambda x: Counter(" ".join(x.dropna()).split()).most_common(20)
        )
        data = []
        for year, tokens in yearly_top_20_tokens.items():
            row = [year]
            for tk, freq in tokens:
                row.append(f"{tk}({freq})")
            data.append(row)
        max_columns = max(len(row) for row in data) if data else 0
        data = [row + [''] * (max_columns - len(row)) for row in data]
        columns = ['Year'] + [f'{i + 1}위' for i in range(max_columns - 1)]
        yearly_top_20_df = pd.DataFrame(data, columns=columns)
        yearly_top_20_df = yearly_top_20_df.sort_values(by='Year', ascending=True)
        yearly_top_20_df.to_excel(writer, sheet_name='연도별 상위 20개 단어', index=False)



    # 5. 특정 단어별 연도별 추이 (TRACKING_WORDS가 비어있지 않을 때만 생성)
    if 'Year' in df.columns and 'Token' in df.columns and TRACKING_WORDS:
        print(f"추적 단어: {TRACKING_WORDS}")
        
        # Token 열에서 각 단어가 포함된 행 확인
        token_lists_for_tracking = df['Token'].apply(lambda x: x.split() if isinstance(x, str) else [])
        
        # 각 단어별 연도별 빈도 계산
        tracking_data = {'Year': sorted(df['Year'].unique())}
        
        for word in TRACKING_WORDS:
            yearly_counts = []
            for year in sorted(df['Year'].unique()):
                year_mask = df['Year'] == year
                year_tokens = token_lists_for_tracking[year_mask]
                # 해당 연도에서 특정 단어가 나타난 횟수 계산
                count = year_tokens.apply(lambda toks: toks.count(word)).sum()
                yearly_counts.append(count)
            tracking_data[word] = yearly_counts
        
        tracking_df = pd.DataFrame(tracking_data)
        
        # 합계 행 추가
        total_row = {'Year': '합계'}
        for word in TRACKING_WORDS:
            total_row[word] = tracking_df[word].sum()
        tracking_df = pd.concat([tracking_df, pd.DataFrame([total_row])], ignore_index=True)
        
        # 엑셀에 저장
        tracking_df.to_excel(writer, sheet_name='단어 추이', index=False)
        worksheet_tracking = writer.sheets['단어 추이']
        
        # 그래프 생성 (각 단어별)
        chart_positions = [f'F{2 + 14 * i}' for i in range(len(TRACKING_WORDS))]
        
        for idx, word in enumerate(TRACKING_WORDS):
            chart = workbook.add_chart({'type': 'line'})
            chart.add_series({
                'name': word,
                'categories': ['단어 추이', 1, 0, len(tracking_df) - 1, 0],
                'values': ['단어 추이', 1, idx + 1, len(tracking_df) - 1, idx + 1],
                'marker': {'type': 'circle', 'size': 7},
            })
            chart.set_title({'name': f'{word} 연도별 빈도'})
            chart.set_x_axis({'name': 'Year'})
            chart.set_y_axis({'name': '빈도'})
            worksheet_tracking.insert_chart(chart_positions[idx], chart)
        
        print("✓ '단어 추이' 시트 생성 완료")
    
    elif TRACKING_WORDS:
        print("⚠ 경고: 'Year' 또는 'Token' 열이 없어서 '단어 추이' 시트를 생성할 수 없습니다.")
    
    else:
        print("ℹ TRACKING_WORDS가 비어있어 '단어 추이' 시트를 생성하지 않습니다.")



print(f"\n✓ 분석 완료!")
print(f"✓ 결과 파일: {output_path}")


✓ openpyxl 엔진으로 성공!

✓ 데이터 로드 완료: 9089 행
추적 단어: ['세계', '한국']
✓ '단어 추이' 시트 생성 완료

✓ 분석 완료!
✓ 결과 파일: authoress_Freq.xlsx


### 키워드별 공기어

In [6]:
import pandas as pd
from collections import Counter
import numpy as np
from scipy.stats import chi2_contingency


df = pd.read_excel('authoress_Token.xlsx')
keywords = ['세계적','한국적','전통적']
exclude_words = {'여류'}  
topn = 20


# 전체 단어 빈도 계산
df['Token'] = df['Token'].fillna('').astype(str)
all_words = Counter()
for tokens in df['Token']:
    all_words.update(tokens.split())
N = sum(all_words.values())



# 결과 저장 변수
tscore_lists = []
chi2_lists = []
combined_lists = []


for key in keywords:
    df_key = df[df['Token'].str.contains(key)]
    co_counter = Counter()
    for tokens in df_key['Token']:
        token_list = tokens.split()
        if key in token_list:
            # 키워드 자신과 제외 단어들을 모두 제거
            neighbors = set(token_list) - {key} - exclude_words
            for word in neighbors:
                co_counter[word] += 1
    key_count = all_words[key]


    tscore_temp = []
    chi2_temp = []
    combined_temp = []


    for word, freq in co_counter.items():
        word_count = all_words[word]
        expected = key_count * word_count / N


        # t-score
        tscore = (freq - expected) / np.sqrt(freq) if freq > 0 else 0
        tscore_temp.append((word, freq, tscore))


        # Chi-square
        obs = np.array([[freq, key_count - freq],
                        [word_count - freq, N - (key_count + word_count - freq)]])
        try:
            chi2, p, dof, ex = chi2_contingency(obs, correction=False)
        except:
            chi2 = 0
        chi2_temp.append((word, freq, chi2))


        # 통합 결과
        combined_temp.append((word, freq, tscore, chi2))


    tscore_temp = sorted(tscore_temp, key=lambda x: x[2], reverse=True)[:topn]
    tscore_temp += [('', 0, 0)] * (topn - len(tscore_temp))
    tscore_lists.append(tscore_temp)


    chi2_temp = sorted(chi2_temp, key=lambda x: x[2], reverse=True)[:topn]
    chi2_temp += [('', 0, 0)] * (topn - len(chi2_temp))
    chi2_lists.append(chi2_temp)


    combined_temp = sorted(combined_temp, key=lambda x: x[2], reverse=True)[:topn]
    combined_temp += [('', 0, 0, 0)] * (topn - len(combined_temp))
    combined_lists.append(combined_temp)


# 컬럼 정의
tscore_columns = ['순위']
chi2_columns = ['순위']
combined_columns = ['순위']


for key in keywords:
    tscore_columns += [key, f"{key}_빈도", f"{key}_t_score"]
    chi2_columns += [key, f"{key}_빈도", f"{key}_Chi2"]
    combined_columns += [key, f"{key}_빈도", f"{key}_t_score", f"{key}_Chi2"]


# 데이터 행 구성 함수
def make_rows(data_lists, metrics_num):
    rows = []
    for i in range(topn):
        row = [i+1]
        for key_idx in range(len(keywords)):
            values = data_lists[key_idx][i]
            row.extend(values[:metrics_num])
        rows.append(row)
    return rows


tscore_rows = make_rows(tscore_lists, 3)
chi2_rows = make_rows(chi2_lists, 3)
combined_rows = make_rows(combined_lists, 4)


# DataFrame 생성
tscore_df = pd.DataFrame(tscore_rows, columns=tscore_columns)
chi2_df = pd.DataFrame(chi2_rows, columns=chi2_columns)
combined_df = pd.DataFrame(combined_rows, columns=combined_columns)


# Excel 저장
with pd.ExcelWriter('authoress_co.xlsx') as writer:
    tscore_df.to_excel(writer, sheet_name='t-score', index=False)
    chi2_df.to_excel(writer, sheet_name='Chi-square', index=False)
    combined_df.to_excel(writer, sheet_name='통합결과', index=False)


print("authoress_co.xlsx 파일에 3개 시트가 저장되었습니다.")


authoress_co.xlsx 파일에 3개 시트가 저장되었습니다.


## 데이터 전처리(R)

In [ ]:
# 1. 패키지 로드
library(tidyverse)
library(readxl)
library(stm)

# 2. 데이터 불러오기 (실행 파일과 같은 폴더)
base_dir <- getwd()
data_file <- file.path(base_dir, "authoress_Token.xlsx")
df <- read_excel(data_file)

# 3. 변수명 확인 (필요시)
print(names(df))
# 예상: Year, Month, Publisher, Title, Token, Voca, ... 등

# 4. 불용어(stopwords) 지정 (필요시 추가/수정)
stopwords <- c('여류')

# 5. (선택) 날짜 변수 합치기
df <- df %>%
  mutate(Date = as.Date(sprintf("%04d-%02d-01", Year, Month)))


# 6. Token 열이 factor면 character로 변환
df$Token <- as.character(df$Token)

# 7. vocab 중복 제거
all_tokens <- unlist(strsplit(df$Token, " "))
vocab <- unique(all_tokens)

# 8. STM용 입력 데이터 생성(textProcessor)
myprocess <- textProcessor(
  documents = df$Token,
  metadata = df,
  wordLengths = c(2, Inf),      
  lowercase = FALSE,              
  removenumbers = FALSE,
  removepunctuation = FALSE,
  removestopwords = FALSE,
  stem = FALSE,
  customstopwords = stopwords
)

# 9. vocab 중복 제거 반영 (myprocess$vocab에 unique 적용)
myprocess$vocab <- unique(myprocess$vocab)

# 10. 3개 이상의 문서에 등장한 단어만 사용
out <- prepDocuments(
  myprocess$documents,
  myprocess$vocab,
  myprocess$meta,
  lower.thresh = 3
)

# 11. 결과 저장
save_dir <- file.path(base_dir, "topic_selection_results", "STM")
if (!dir.exists(save_dir)) dir.create(save_dir, recursive = TRUE)
saveRDS(out, file = file.path(save_dir, "out_preprocessed.rds"))

cat("STM용 데이터 전처리 및 저장 완료!\n")

Warning message:
"package 'tidyverse' was built under R version 4.2.3"
Warning message:
"package 'tibble' was built under R version 4.2.3"
Warning message:
"package 'tidyr' was built under R version 4.2.3"
Warning message:
"package 'readr' was built under R version 4.2.3"
Warning message:
"package 'purrr' was built under R version 4.2.3"
Warning message:
"package 'dplyr' was built under R version 4.2.3"
Warning message:
"package 'stringr' was built under R version 4.2.3"
Warning message:
"package 'forcats' was built under R version 4.2.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<ht

[1] "Year"      "Month"     "Day"       "Publisher" "Title"     "Contents" 
[7] "Token"    
Building corpus... 
Remove Custom Stopwords...
Creating Output... 


: 

## 토픽수 탐색 searchK(R)

In [1]:
# ===============================================
# [설정 섹션] 사용자 커스터마이징 구간
# ===============================================

# 토픽 수 범위 설정 (K_min:K_max)
K_min <- 10
K_max <- 20

# 시드값 (재현성 보장)
SEED_VALUE <- 2024

# EM 알고리즘 최대 반복 횟수
MAX_EM_ITS <- 75

# 병렬 처리 코어 수 (1 = 단일 코어, >1 = 병렬 처리)
N_CORES <- 1

# 그래프 해상도 (DPI)
GRAPH_DPI <- 300

# 그래프 크기 설정
GRAPH_WIDTH <- 12
GRAPH_HEIGHT <- 9

# =================================================

# 1. 패키지 로드
library(stm)
library(tidyverse)
library(ggplot2)
library(gridExtra)
library(rlang)

# 2. 파일 경로 설정
base_dir <- getwd()
searchK_rds_path <- file.path(base_dir, "topic_selection_results", "STM", "model_searchK.rds")
out_rds_path <- file.path(base_dir, "topic_selection_results", "STM", "out_preprocessed.rds")
output_dir <- file.path(base_dir, "topic_selection_results", "STM")

# output_dir 생성 (없으면)
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

# ====================================================
# 3. searchK 결과 불러오기 또는 새로 계산
# ====================================================
if (file.exists(searchK_rds_path)) {
  message("저장된 searchK 결과를 불러옵니다.")
  model_searchK <- readRDS(searchK_rds_path)
} else {
  message("searchK 결과 파일이 없어 새로 계산합니다.")
  out <- readRDS(out_rds_path)
  out$meta$Date <- as.Date(out$meta$Date)
  out$meta$Year <- as.numeric(format(out$meta$Date, "%Y"))
  
  # Publisher, Voca 컬럼 존재 여부에 따라 prevalence 공식 결정
  if ("Publisher" %in% colnames(out$meta) & "Voca" %in% colnames(out$meta)) {
    prevalence_formula <- ~Publisher + Voca + s(Year)
    out$meta$Publisher <- as.factor(out$meta$Publisher)
    out$meta$Voca <- as.factor(out$meta$Voca)
    message("Prevalence formula: ~Publisher + Voca + s(Year)")
  } else if ("Publisher" %in% colnames(out$meta)) {
    prevalence_formula <- ~Publisher + s(Year)
    out$meta$Publisher <- as.factor(out$meta$Publisher)
    message("Prevalence formula: ~Publisher + s(Year)")
  } else if ("Voca" %in% colnames(out$meta)) {
    prevalence_formula <- ~Voca + s(Year)
    out$meta$Voca <- as.factor(out$meta$Voca)
    message("Prevalence formula: ~Voca + s(Year)")
  } else {
    prevalence_formula <- ~s(Year)
    message("Prevalence formula: ~s(Year)")
  }
  
  # 설정 섹션에서 정의한 K 범위 사용
  k_candidates <- K_min:K_max
  message(sprintf("searchK 실행: K = %s", paste(k_candidates, collapse = ", ")))
  
  model_searchK <- searchK(
    documents = out$documents,
    vocab = out$vocab,
    K = k_candidates,
    prevalence = prevalence_formula,
    data = out$meta,
    init.type = "Spectral",
    seed = SEED_VALUE,
    max.em.its = MAX_EM_ITS,
    cores = N_CORES
  )
  
  saveRDS(model_searchK, searchK_rds_path)
  message(sprintf("searchK 결과가 저장되었습니다: %s", searchK_rds_path))
}

# ===================================================
# 4. 결과 데이터프레임 정리 및 타입 변환
# ===================================================
results_df <- model_searchK$results %>%
  as_tibble() %>%
  mutate(
    K = as.integer(K),
    semcoh = as.numeric(semcoh),
    exclus = as.numeric(exclus),
    heldout = as.numeric(heldout),
    residual = as.numeric(residual),
    bound = as.numeric(bound)
  )

message("결과 데이터프레임 구조:")
print(str(results_df))
message("\n결과 데이터프레임 (처음 5행):")
print(head(results_df, 5))

# ==================================================
# 5. ggplot2 + gridExtra로 4분할 그래프 생성
#    (Lower Bound 제외, Exclusivity 포함)
# ==================================================

# 5-1. Held-Out Likelihood 그래프
p1 <- ggplot(results_df, aes(x = K, y = heldout)) +
  geom_line(color = "#2E86AB", size = 1) + 
  geom_point(color = "#2E86AB", size = 3) +
  labs(title = "Held-Out Likelihood",
       x = "Number of Topics (K)",
       y = "Likelihood") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 12, face = "bold"),
    panel.grid.minor = element_blank(),
    axis.text = element_text(size = 10)
  ) +
  scale_x_continuous(breaks = unique(results_df$K))

# 5-2. Residuals 그래프
p2 <- ggplot(results_df, aes(x = K, y = residual)) +
  geom_line(color = "#A23B72", size = 1) + 
  geom_point(color = "#A23B72", size = 3) +
  labs(title = "Residuals",
       x = "Number of Topics (K)",
       y = "Residuals") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 12, face = "bold"),
    panel.grid.minor = element_blank(),
    axis.text = element_text(size = 10)
  ) +
  scale_x_continuous(breaks = unique(results_df$K))

# 5-3. Semantic Coherence 그래프
p3 <- ggplot(results_df, aes(x = K, y = semcoh)) +
  geom_line(color = "#F18F01", size = 1) + 
  geom_point(color = "#F18F01", size = 3) +
  labs(title = "Semantic Coherence",
       x = "Number of Topics (K)",
       y = "Coherence") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 12, face = "bold"),
    panel.grid.minor = element_blank(),
    axis.text = element_text(size = 10)
  ) +
  scale_x_continuous(breaks = unique(results_df$K))

# 5-4. Exclusivity 그래프
p4 <- ggplot(results_df, aes(x = K, y = exclus)) +
  geom_line(color = "#C73E1D", size = 1) + 
  geom_point(color = "#C73E1D", size = 3) +
  labs(title = "Exclusivity",
       x = "Number of Topics (K)",
       y = "Exclusivity") +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 12, face = "bold"),
    panel.grid.minor = element_blank(),
    axis.text = element_text(size = 10)
  ) +
  scale_x_continuous(breaks = unique(results_df$K))

# 5-5. 4분할 그래프를 하나의 이미지로 저장
combined_plot_path <- file.path(output_dir, "searchK_diagnostic_4metrics.png")

g <- arrangeGrob(p1, p2, p3, p4, 
                 ncol = 2, nrow = 2,
                 top = "Diagnostic Values by Number of Topics")

ggsave(filename = combined_plot_path, 
       plot = g, 
       width = GRAPH_WIDTH, 
       height = GRAPH_HEIGHT,
       dpi = GRAPH_DPI)

message(sprintf("4분할 진단 그래프가 저장되었습니다: %s", combined_plot_path))

# ===============================================
# 6. 개별 그래프 저장 (선택사항: 각 지표별 고해상도 그래프)
# ===============================================
metrics <- c("semcoh", "exclus", "heldout", "residual")
metric_labels <- c(
  "semcoh" = "Semantic Coherence",
  "exclus" = "Exclusivity",
  "heldout" = "Held-Out Likelihood",
  "residual" = "Residuals"
)

for (metric in metrics) {
  p <- ggplot(results_df, aes(x = K, y = .data[[metric]])) +
    geom_line(color = "#2E86AB", size = 1) + 
    geom_point(color = "#2E86AB", size = 3) +
    ggtitle(paste("SearchK Diagnostic:", metric_labels[metric])) +
    labs(x = "Number of Topics (K)", y = metric_labels[metric]) +
    theme_bw() +
    theme(
      plot.title = element_text(hjust = 0.5, size = 13, face = "bold"),
      panel.grid.minor = element_blank(),
      axis.text = element_text(size = 11)
    ) +
    scale_x_continuous(breaks = unique(results_df$K))
  
  individual_plot_path <- file.path(output_dir, paste0("searchK_", metric, ".png"))
  ggsave(filename = individual_plot_path,
         plot = p, 
         width = 9, 
         height = 6,
         dpi = GRAPH_DPI)
  
  message(sprintf("  ✓ %s 그래프 저장됨: %s", metric_labels[metric], individual_plot_path))
}

# =============================================
# 7. 분석 결과 요약 테이블 저장
# =============================================

# 최적값 찾기 (방향성 고려)
results_summary <- results_df %>%
  mutate(
    # Held-out likelihood: 높을수록 좋음 → 정규화 (높은 값을 높은 스코어로)
    score_heldout = scale(heldout, center = TRUE, scale = TRUE)[, 1],
    
    # Residuals: 낮을수록 좋음 → 반전 정규화
    score_residual = -scale(residual, center = TRUE, scale = TRUE)[, 1],
    
    # Semantic Coherence: 높을수록 좋음 (음수, 0에 가까울수록 좋음) → 정규화
    score_semcoh = scale(semcoh, center = TRUE, scale = TRUE)[, 1],
    
    # Exclusivity: 높을수록 좋음 → 정규화
    score_exclus = scale(exclus, center = TRUE, scale = TRUE)[, 1]
  ) %>%
  # 4가지 지표의 가중 평균 (동등 가중치)
  mutate(
    composite_score = (score_heldout + score_residual + score_semcoh + score_exclus) / 4
  ) %>%
  select(K, semcoh, exclus, heldout, residual, composite_score) %>%
  arrange(desc(composite_score))

message("\n=== 종합 분석 결과 (Composite Score 기준, 상위 5개) ===")
print(head(results_summary, 5))

# 결과 테이블 CSV 저장
results_summary_path <- file.path(output_dir, "searchK_summary_table.csv")
write_csv(results_summary, results_summary_path)
message(sprintf("\n결과 테이블이 저장되었습니다: %s", results_summary_path))


SyntaxError: invalid syntax. Perhaps you forgot a comma? (1020701489.py, line 47)

## 토픽별 키워드와 비중(R)

In [ ]:
# 1. 패키지 로드
library(openxlsx)
library(ggplot2)
library(dplyr)
library(tibble)
library(stm)

# 2. 저장 폴더 및 데이터 경로 설정
base_dir <- getwd()
K_values <- c(18,15,17)

# 3. 전처리된 STM 데이터 불러오기
out <- readRDS(file.path(base_dir, "topic_selection_results", "STM", "out_preprocessed.rds"))

# 4. 메타데이터 컬럼 타입 변환
# (Publisher, Voca 등 팩터 처리)
if ("Publisher" %in% colnames(out$meta)) {
  out$meta$Publisher <- as.factor(out$meta$Publisher)
}
if ("Voca" %in% colnames(out$meta)) {
  out$meta$Voca <- as.factor(out$meta$Voca)
}
# Date는 stm 내부에서 s() 함수 사용 시 numeric으로 변환되어야 안전합니다
if ("Date" %in% colnames(out$meta)) {
  out$meta$numeric_date <- as.numeric(as.Date(out$meta$Date))
}

# 5. prevalence 공식 자동 생성 (안전한 방식)
prevalence_vars <- c()
if ("Publisher" %in% colnames(out$meta)) prevalence_vars <- c(prevalence_vars, "Publisher")
if ("Voca" %in% colnames(out$meta)) prevalence_vars <- c(prevalence_vars, "Voca")
# Date 대신 numeric_date 사용 권장
if ("Date" %in% colnames(out$meta)) prevalence_vars <- c(prevalence_vars, "s(numeric_date)")

# 변수가 하나도 없을 경우 NULL 처리 (단순 토픽 모델링)
if (length(prevalence_vars) > 0) {
  prevalence_formula_str <- paste("~", paste(prevalence_vars, collapse = " + "))
  prevalence_formula <- as.formula(prevalence_formula_str)
} else {
  prevalence_formula <- NULL
}

# 6~13 루프 처리 시작
for (optimal_K in K_values) {
  
  cat("\n==================================\n")
  cat("Processing K =", optimal_K, "...\n")
  
  save_dir <- file.path(base_dir, paste0("STM_", optimal_K))
  if (!dir.exists(save_dir)) dir.create(save_dir, recursive = TRUE)
  
  # STM 모델 불러오기 또는 학습
  stm_model_path <- file.path(save_dir, "stm_model.rds")
  if (file.exists(stm_model_path)) {
    message("  > 기존 모델 로드 중...")
    stm_model <- readRDS(stm_model_path)
  } else {
    message("  > 새 모델 학습 중...")
    stm_model <- stm(
      documents = out$documents,
      vocab = out$vocab,
      K = optimal_K,
      prevalence = prevalence_formula,
      data = out$meta,
      init.type = "Spectral",
      seed = 2024,
      max.em.its = 75,
      verbose = FALSE
    )
    saveRDS(stm_model, stm_model_path)
  }
  
  # 토픽별 키워드 추출
  topic_labels <- labelTopics(stm_model, n = 10)
  keyword_types <- c("prob", "frex", "lift", "score")
  
  # Prob, FREX 표 저장용 변환
  prob_keywords <- as.data.frame(topic_labels$prob)
  rownames(prob_keywords) <- paste0("Topic_", 1:nrow(prob_keywords))
  colnames(prob_keywords) <- paste0("Rank_", 1:ncol(prob_keywords))
  
  frex_keywords <- as.data.frame(topic_labels$frex)
  rownames(frex_keywords) <- paste0("Topic_", 1:nrow(frex_keywords))
  colnames(frex_keywords) <- paste0("Rank_", 1:ncol(frex_keywords))
  
  # 키워드 기여도(확률)
  beta_matrix <- exp(stm_model$beta$logbeta[[1]])
  vocab <- stm_model$vocab
  n_rank <- ncol(topic_labels$prob)
  keyword_contrib_long <- list()
  
  for (k in 1:optimal_K) {
    for (type in keyword_types) {
      terms <- topic_labels[[type]][k, ]
      idx <- match(terms, vocab)
      
      # 안전장치: idx가 NA인 경우 (드물지만 발생 가능) 처리
      valid_idx <- !is.na(idx)
      current_terms <- terms[valid_idx]
      current_probs <- beta_matrix[k, idx[valid_idx]]
      
      # 텍스트 포맷팅
      formatted_vals <- sprintf("%s (%.4f)", current_terms, current_probs)
      
      # 길이가 부족할 경우 채워넣기 (안전장치)
      if(length(formatted_vals) < n_rank) {
        formatted_vals <- c(formatted_vals, rep("", n_rank - length(formatted_vals)))
      }
      
      rank_cols <- setNames(as.list(formatted_vals), paste0("Rank_", 1:n_rank))
      
      keyword_contrib_long[[length(keyword_contrib_long) + 1]] <- 
        data.frame(
          Topic = paste0("Topic_", k),
          Type = type,
          rank_cols,
          stringsAsFactors = FALSE
        )
    }
  }
  keyword_contrib_df <- do.call(rbind, keyword_contrib_long) %>%
    arrange(Topic, factor(Type, levels = keyword_types))
  
  # 토픽 비중 계산
  topic_matrix <- stm_model[["theta"]]
  average_topic_proportions <- colMeans(topic_matrix)
  topic_proportion <- data.frame(
    Topic = paste0("Topic_", 1:optimal_K),
    Proportion = average_topic_proportions
  )
  topic_proportion$Percent <- sprintf("%.2f%%", topic_proportion$Proportion * 100)
  topic_proportion_sorted <- topic_proportion %>% arrange(desc(Proportion))
  
  # 토픽별 상위 20 문서 추출
  top_docs_list <- lapply(1:optimal_K, function(k) {
    top_idx <- order(topic_matrix[, k], decreasing = TRUE)[1:20]
    data.frame(
      Topic = paste0("Topic_", k),
      Rank = 1:20,
      Proportion = topic_matrix[top_idx, k],
      Date = if ("Date" %in% colnames(out$meta)) out$meta$Date[top_idx] else NA,
      Title = if ("Title" %in% colnames(out$meta)) out$meta$Title[top_idx] else "No Title",
      Publisher = if ("Publisher" %in% colnames(out$meta)) out$meta$Publisher[top_idx] else NA,
      Voca = if ("Voca" %in% colnames(out$meta)) out$meta$Voca[top_idx] else NA,
      stringsAsFactors = FALSE
    )
  })
  top_docs_df <- bind_rows(top_docs_list)
  
  # 엑셀 저장
  output_file <- file.path(save_dir, paste0("STM_", optimal_K, ".xlsx"))
  wb <- createWorkbook()
  
  addWorksheet(wb, "토픽별_비중")
  writeData(wb, "토픽별_비중", "1. 토픽 번호순", startRow = 1, startCol = 1)
  writeData(wb, "토픽별_비중", topic_proportion, startRow = 2, rowNames = FALSE)
  start_row_sorted <- nrow(topic_proportion) + 4
  writeData(wb, "토픽별_비중", "2. 비중 내림차순", startRow = start_row_sorted - 1, startCol = 1)
  writeData(wb, "토픽별_비중", topic_proportion_sorted, startRow = start_row_sorted, rowNames = FALSE)
  
  addWorksheet(wb, "토픽별_Prob")
  writeData(wb, "토픽별_Prob", prob_keywords, rowNames = TRUE)
  
  addWorksheet(wb, "토픽별_FREX")
  writeData(wb, "토픽별_FREX", frex_keywords, rowNames = TRUE)
  
  addWorksheet(wb, "키워드 기여도")
  writeData(wb, "키워드 기여도", keyword_contrib_df)
  
  addWorksheet(wb, "토픽별 상위 20문서")
  writeData(wb, "토픽별 상위 20문서", top_docs_df)
  
  saveWorkbook(wb, output_file, overwrite = TRUE)
  message(paste("  > 엑셀 저장 완료:", output_file))
  
  # 효과추정 (estimateEffect) - 수정된 부분
  if (!is.null(prevalence_formula)) {
    # prevalence formula의 우변(독립변수)만 가져와서 수식 재조합
    rhs_formula <- as.character(prevalence_formula)[2]
    effect_formula <- as.formula(paste("1:optimal_K ~", rhs_formula))
    
    message("  > 효과 추정(EstimateEffect) 계산 중...")
    stm_effect_model <- estimateEffect(
      effect_formula,
      stm_model,
      meta = out$meta,
      uncertainty = "Global",
      prior = 1e-5
    )
    saveRDS(stm_effect_model, file.path(save_dir, "stm_effect_model.rds"))
    message("  > 효과 추정 모델 저장 완료")
  } else {
    message("  > 공변량(Prevalence)이 없어 효과 추정을 건너뜁니다.")
  }
  
  cat("모델 K =", optimal_K, "처리 완료\n")
}


Processing K = 18 ...


  > 새 모델 학습 중...

  > 엑셀 저장 완료: d:/text mining/bioethics2012/STM_18/STM_18.xlsx

  > 효과 추정(EstimateEffect) 계산 중...

  > 효과 추정 모델 저장 완료



모델 K = 18 처리 완료

Processing K = 15 ...


  > 새 모델 학습 중...

  > 엑셀 저장 완료: d:/text mining/bioethics2012/STM_15/STM_15.xlsx

  > 효과 추정(EstimateEffect) 계산 중...

  > 효과 추정 모델 저장 완료



모델 K = 15 처리 완료

Processing K = 17 ...


  > 새 모델 학습 중...

  > 엑셀 저장 완료: d:/text mining/bioethics2012/STM_17/STM_17.xlsx

  > 효과 추정(EstimateEffect) 계산 중...

  > 효과 추정 모델 저장 완료



모델 K = 17 처리 완료


## 토픽 비중 추이 (R)

In [4]:
library(stm)
library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(readr)
library(scales)
library(openxlsx)

# ====== 사용자 설정 영역 (여기서만 수정) ======
K_list <- c(18,15,17)    # 실행할 토픽 수 목록 예(10,11,12)
year_min <- 2010             # 분석 시작 연도
year_max <- 2025             # 분석 종료 연도
year_breaks <- 5             # X축 연도 라벨 간격

# ====== 그래프 폰트 크기 설정 ======
font_size_title_all <- 22     # 전체 그래프 제목 폰트
font_size_base_all <- 20      # 전체 그래프 기본 폰트 (축, 범례 등)
font_size_axis_all <- 16      # 전체 그래프 축 눈금 폰트

font_size_title_topic <- 22   # 개별 토픽 그래프 제목 폰트
font_size_base_topic <- 18    # 개별 토픽 그래프 기본 폰트
font_size_axis_topic <- 14    # 개별 토픽 그래프 축 눈금 폰트

# ====== 반복 처리 시작 (모든 K에 대해 실행) ======
for (optimal_K in K_list) {
  
  # ====== 폴더 설정 ======
  base_dir <- file.path(getwd(), paste0("STM_", optimal_K))
  date_dir <- file.path(base_dir, "Trend")
  if (!dir.exists(date_dir)) dir.create(date_dir, recursive = TRUE)
  
  # ====== 모델 및 데이터 불러오기 ======
  stm_model <- readRDS(file.path(base_dir, "stm_model.rds"))
  out <- readRDS(file.path(getwd(), "topic_selection_results", "STM", "out_preprocessed.rds"))
  meta <- out$meta
  
  # ====== 문서별 토픽 분포와 연도 결합 ======
  theta_df <- as.data.frame(stm_model$theta)[, 1:optimal_K]
  colnames(theta_df) <- paste0("Topic_", 1:optimal_K)
  theta_df$Year <- year(meta$Date)
  
  # ====== 전체 누적 면적 그래프용 연도별 평균 비중 ======
  topic_year_long <- theta_df %>%
    group_by(Year) %>%
    summarise(across(starts_with("Topic_"), mean, na.rm = TRUE)) %>%
    filter(Year >= year_min, Year <= year_max) %>%
    pivot_longer(cols = starts_with("Topic_"),
                 names_to = "Topic",
                 values_to = "Proportion") %>%
    arrange(Topic, Year)
  
  topic_year_long$Topic_label <- factor(topic_year_long$Topic,
                                        levels = paste0("Topic_", 1:optimal_K))
  
  # ====== 전체 누적 면적 그래프 (연도별) ======
  p_area <- ggplot(topic_year_long, aes(x = Year, y = Proportion, fill = Topic_label)) +
    geom_area(position = "fill", color = "white", size = 0.2, alpha = 0.95) +
    scale_y_continuous(labels = percent_format(accuracy = 1), expand = c(0, 0)) +
    scale_x_continuous(breaks = seq(year_min, year_max, by = year_breaks), 
                       limits = c(year_min, year_max), expand = c(0, 0)) +
    labs(title = "연도별 토픽 비중 (100% 누적 면적 그래프)",
         x = "연도", y = "비중(%)") +
    theme_minimal(base_size = font_size_base_all) +
    theme(plot.title = element_text(face = "bold", size = font_size_title_all),
          axis.title = element_text(size = font_size_base_all),
          axis.text.x = element_text(angle = 45, hjust = 1, size = font_size_axis_all),
          axis.text.y = element_text(size = font_size_axis_all),
          legend.text = element_text(size = font_size_base_all - 2),
          legend.title = element_text(size = font_size_base_all),
          plot.background = element_rect(fill = "white", color = NA),
          panel.background = element_rect(fill = "white", color = NA),
          legend.background = element_rect(fill = "white", color = NA),
          legend.box.background = element_rect(fill = "white", color = NA))
  
  ggsave(filename = file.path(date_dir, 
                              paste0("연도별_토픽비중_전체_", year_min, "_", year_max, ".png")),
         plot = p_area, width = 12, height = 7, dpi = 300, bg = "white")
  
  # ====== 개별 토픽별 꺾은선 그래프 (연도별, y축 0% 고정) ======
  for (k in 1:optimal_K) {
    topic_name <- paste0("Topic_", k)
    topic_data <- topic_year_long %>% filter(Topic_label == topic_name)
    
    p_topic <- ggplot(topic_data, aes(x = Year, y = Proportion, group = 1)) +
      geom_line(color = "#2C3E50", linewidth = 1) +
      geom_point(color = "#2980B9", size = 2) +
      scale_x_continuous(breaks = seq(year_min, year_max, by = year_breaks), 
                         limits = c(year_min, year_max)) +
      scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, NA)) +
      labs(title = topic_name,
           x = "연도", y = "평균 비중(%)") +
      theme_minimal(base_size = font_size_base_topic) +
      theme(plot.title = element_text(face = "bold", size = font_size_title_topic),
            axis.title = element_text(size = font_size_base_topic),
            axis.text.x = element_text(angle = 45, hjust = 1, size = font_size_axis_topic),
            axis.text.y = element_text(size = font_size_axis_topic))
    
    ggsave(filename = file.path(date_dir, paste0("연도별_토픽비중_", topic_name, ".png")),
           plot = p_topic, width = 8, height = 5, dpi = 300, bg = "white")
  }
  
  # ====== 연도별 토픽 비중 데이터 엑셀 저장 ======
  wb <- createWorkbook()
  addWorksheet(wb, "연도별_토픽비중_데이터")
  writeData(wb, "연도별_토픽비중_데이터", topic_year_long)
  excel_filename <- paste0("STM_", optimal_K, "_yearly.xlsx")
  saveWorkbook(wb, file.path(date_dir, excel_filename), overwrite = TRUE)
  
  cat("\n✓ K=", optimal_K, " 분석 완료!\n", sep = "")
}

cat("\n====== 모든 분석 완료 ======\n")



Attaching package: 'scales'


The following object is masked from 'package:purrr':

    discard


The following object is masked from 'package:readr':

    col_factor


Warning message:
"There was 1 warning in `summarise()`.
ℹ In argument: `across(starts_with("Topic_"), mean, na.rm = TRUE)`.
ℹ In group 1: `Year = 2010`.
Caused by warning:
! The `...` argument of `across()` is deprecated as of dplyr 1.1.0.
Supply arguments directly to `.fns` through an anonymous function instead.

  # Previously
  across(a:b, mean, na.rm = TRUE)

  # Now
  across(a:b, \(x) mean(x, na.rm = TRUE))"



✓ K=18 분석 완료!

✓ K=15 분석 완료!

✓ K=17 분석 완료!

====== 모든 분석 완료 ======


### 언론사별 효과추정(R)

In [5]:
library(stm)
library(dplyr)
library(ggplot2)
library(openxlsx)
library(scales)
library(tidyr)

# ====== 토픽 개수 설정 ======
optimal_K <- 16

# ====== 폴더 설정 및 생성 ======
base_dir <- file.path(getwd(), paste0("STM_", optimal_K))
publisher_dir <- file.path(base_dir, "Publisher")
barplot_dir <- file.path(publisher_dir, "barplot")
diffplot_dir <- file.path(publisher_dir, "differenceplot")
if (!dir.exists(barplot_dir)) dir.create(barplot_dir, recursive = TRUE)
if (!dir.exists(diffplot_dir)) dir.create(diffplot_dir, recursive = TRUE)

# ====== 모델 및 데이터 불러오기 ======
stm_model <- readRDS(file.path(base_dir, "stm_model.rds"))
out <- readRDS(file.path(getwd(), "topic_selection_results", "STM", "out_preprocessed.rds"))
meta <- out$meta

# ====== 문서별 토픽 분포와 신문사 결합 ======
theta_df <- as.data.frame(stm_model$theta)
colnames(theta_df) <- paste0("Topic_", 1:optimal_K)
theta_df$Publisher <- meta$Publisher

# ====== 신문사별 토픽별 평균 비중 계산 (barplot용) ======
topic_publisher_long <- theta_df %>%
  group_by(Publisher) %>%
  summarise(across(starts_with("Topic_"), mean, na.rm = TRUE)) %>%
  pivot_longer(cols = starts_with("Topic_"),
               names_to = "Topic",
               values_to = "Proportion") %>%
  filter(!is.na(Publisher)) %>%
  arrange(Topic, Publisher)

# ====== 토픽별 신문사 평균 비중 barplot 저장 (각 토픽별) ======
for (k in 1:optimal_K) {
  topic_name <- paste0("Topic_", k)
  pub_data <- topic_publisher_long %>% filter(Topic == topic_name)
  p_pub <- ggplot(pub_data, aes(x = Publisher, y = Proportion, fill = Publisher)) +
    geom_col(show.legend = FALSE) +
    scale_y_continuous(labels = percent_format(accuracy = 1)) +
    labs(title = paste0("토픽별 신문사 비중 (", topic_name, ")"),
         x = "신문사", y = "비중(%)") +
    theme_minimal() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      panel.background = element_rect(fill = "white", color = NA),
      plot.background = element_rect(fill = "white", color = NA)
    )
  ggsave(filename = file.path(barplot_dir, paste0("토픽별_신문사비중_barplot_", topic_name, ".png")),
         plot = p_pub, width = 8, height = 5, dpi = 300, bg = "white")
}

# ====== barplot 데이터 엑셀 저장 ======
wb_bar <- createWorkbook()
addWorksheet(wb_bar, "신문사별_토픽비중_데이터")
writeData(wb_bar, "신문사별_토픽비중_데이터", topic_publisher_long)
excel_filename_bar <- paste0("STM_", optimal_K, "_publisher_barplot.xlsx")
saveWorkbook(wb_bar, file.path(barplot_dir, excel_filename_bar), overwrite = TRUE)

# ====== difference plot (엑셀 파일이 있으면 불러오고, 없으면 계산/저장) ======
excel_path <- file.path(diffplot_dir, "differenceplot_all_publishers.xlsx")
unique_publishers <- unique(meta$Publisher)

if (file.exists(excel_path)) {
  sheet_names <- getSheetNames(excel_path)
  result_list <- lapply(sheet_names, function(sheet) {
    read.xlsx(excel_path, sheet = sheet)
  })
  names(result_list) <- sheet_names
} else {
  wb <- createWorkbook()
  result_list <- list()
  for (pub in unique_publishers) {
    meta$Publisher_group <- ifelse(meta$Publisher == pub, pub, "전체")
    meta$Publisher_group <- factor(meta$Publisher_group, levels = c(pub, "전체"))
    stm_effect_model <- estimateEffect(1:optimal_K ~ Publisher_group, stm_model, meta = meta, uncertainty = "Global")
    eff_summary <- summary(stm_effect_model, topics = 1:optimal_K)
    result_df <- lapply(eff_summary$tables, function(tab) {
      data.frame(
        Estimate = tab[2, "Estimate"],
        StdError = tab[2, "Std. Error"],
        p_value = tab[2, "Pr(>|t|)"]
      )
    }) %>% bind_rows()
    result_df$Topic <- paste0("Topic ", 1:optimal_K)
    result_df$stars <- cut(result_df$p_value,
      breaks = c(-Inf, 0.001, 0.01, 0.05, Inf),
      labels = c("***", "**", "*", "")
    )
    result_df$Topic <- factor(result_df$Topic, levels = paste0("Topic ", 1:optimal_K))
    result_df <- result_df[, c("Topic", "stars", "p_value", "Estimate", "StdError")]
    addWorksheet(wb, pub)
    writeData(wb, pub, result_df)
    result_list[[pub]] <- result_df
  }
  saveWorkbook(wb, excel_path, overwrite = TRUE)
}


Warning message in gzfile(file, "rb"):
"cannot open compressed file 'd:/text mining/bioethics2012/STM_16/stm_model.rds', probable reason 'No such file or directory'"


ERROR: Error in gzfile(file, "rb"): cannot open the connection


### 언론사별 그래프 그리기(R)

In [ ]:
# ====== 패키지 로드 ======
library(ggplot2)
library(dplyr)
library(openxlsx)

# ====== 토픽 개수 및 경로 설정 ======
optimal_K <- 16
base_dir <- file.path(getwd(), paste0("STM_", optimal_K))
publisher_dir <- file.path(base_dir, "Publisher")
diffplot_dir <- file.path(publisher_dir, "differenceplot")

# ====== difference plot 데이터 불러오기 ======
excel_path <- file.path(diffplot_dir, "differenceplot_all_publishers.xlsx")
if (!file.exists(excel_path)) stop("differenceplot_all_publishers.xlsx 파일이 존재하지 않습니다.")

sheet_names <- getSheetNames(excel_path)
result_list <- lapply(sheet_names, function(sheet) {
  read.xlsx(excel_path, sheet = sheet)
})
names(result_list) <- sheet_names

# ====== difference plot 그리기 ======
for (pub in names(result_list)) {
  result_df <- result_list[[pub]]
  
  # 토픽 번호 추출 및 오름차순 정렬
  result_df <- result_df %>%
    mutate(Topic_num = as.numeric(gsub("Topic ", "", Topic))) %>%
    arrange(Topic_num)
  
  # y축에 순서 반영
  p <- ggplot(result_df, aes(x = Estimate, y = reorder(Topic, Topic_num))) +
    geom_point(size = 2) +
    geom_errorbarh(aes(xmin = Estimate - 1.96 * StdError, xmax = Estimate + 1.96 * StdError), height = 0.2) +
    geom_vline(xintercept = 0, linetype = "dashed") +
    geom_text(aes(label = stars), color = "red", hjust = -0.7, size = 6) +
    xlab(paste0(pub, " <------------------------> 전체 언론사 평균")) +
    ylab("토픽") +
    scale_x_continuous(
      limits = c(-0.29, 0.1),
      breaks = c(-0.2, -0.1, 0, 0.05)
    ) + 
    theme_minimal(base_family = "NanumGothic") +
    theme(
      axis.text.x = element_text(size = 14),
      axis.text.y = element_text(size = 14, lineheight = 0.7),
      axis.title.x = element_text(size = 18, margin = margin(t = 30)),
      axis.title.y = element_text(size = 16)
    ) +
    ggtitle(paste0("Difference Plot: ", pub, " vs 전체평균"))
  ggsave(filename = file.path(diffplot_dir, paste0("differenceplot_", pub, "_vs_all_with_pvalue.png")),
         plot = p, width = 8, height = 6, dpi = 150, bg = "white")
}


Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_errorbarh()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_errorbarh()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_errorbarh()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_errorbarh()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_errorbarh()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_errorbarh()`)."
